# 01 — Data Exploration

**Goal:** look at the data *before* any model exists, so we know what we are training on.

This notebook answers five questions:

1. What do the scans actually look like? (random samples per class)
2. How many images are in each class? (counts)
3. Is the dataset balanced? (distribution plot)
4. Are all images the same size? (dimensions)
5. What does *class imbalance* mean and why should we care here?

---

## Reproducibility

Notebooks are exploratory, but the numbers in here still need to be reproducible, so the
seed is fixed. Change `SEED` and every random sample below changes with it.

| Setting | Value |
| --- | --- |
| `SEED` | `42` |

**Library versions** (re-run the cell below to refresh):

```
python      3.13
numpy       2.5.3
pandas      3.0.5
matplotlib  3.11.2
seaborn     0.13.2
pillow      12.3.0
torch       2.14.0   (not required for this notebook)
```

> **Kernel:** run this with the project virtualenv, not the system Python
> (see `notebooks/README.md`).

In [ ]:
# ---------------------------------------------------------------------------
# Setup: libraries, seed, and the few paths this notebook needs.
# ---------------------------------------------------------------------------

import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image

# Reproducibility: one seed drives python, numpy and pandas sampling.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("python     ", sys.version.split()[0])
for mod in (np, pd, sns, Image):
    print(f"{mod.__name__:<11}", getattr(mod, "__version__", "n/a"))

# A fixed style so every plot in the project looks the same.
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.bbox"] = "tight"

# What counts as an image. Anything else in a class folder is ignored.
IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp",
}

# The classes this project targets (Kaggle "Brain Tumor MRI Dataset").
# The notebook reads whatever it finds on disk -- this list is only used
# to warn you if a class is missing, never to invent one.
EXPECTED_CLASSES = ["glioma", "meningioma", "pituitary", "no_tumor"]

# The size src/data/dataset.py resizes every image to before training.
TARGET_SIZE = 224

print("\nseed =", SEED, "| target size =", f"{TARGET_SIZE}x{TARGET_SIZE}")

## 1. Find the data

The notebook should not care where you launched it from, so we walk **up** from the
current working directory looking for a folder that contains both `train/` and `test/`
subfolders. Candidates, in order of preference:

```
data/processed/train/    <-- preferred (data/README.md calls this "model-ready")
data/processed/test/
data/train/
data/test/
```

If nothing is found the notebook tells you what to do and the remaining cells degrade
to empty results instead of crashing — so you can read and run this notebook *before*
the dataset has been downloaded.

Expected layout (one folder per class):

```
data/processed/
├── train/
│   ├── glioma/       image1.jpg, image2.jpg, ...
│   ├── meningioma/   ...
│   ├── pituitary/    ...
│   └── no_tumor/     ...
└── test/
    └── ...same four classes...
```

In [ ]:
# ---------------------------------------------------------------------------
# Locate the dataset, then build a manifest: one row per image.
# ---------------------------------------------------------------------------
# The manifest is the backbone of the whole notebook. It is a plain DataFrame
# with one row per image file, so everything downstream is just group-by / plot
# instead of a pile of nested loops over folders.


def find_data_root(start: Path, max_up: int = 5) -> Path | None:
    """Walk up from `start` looking for a folder holding train/ and test/.

    Returns the folder, or None if the dataset is not on disk yet.
    """
    here = start.resolve()
    for _ in range(max_up + 1):
        # data/processed first (see data/README.md), then bare data/, then here.
        for candidate in (here / "data" / "processed", here / "data", here):
            if (candidate / "train").is_dir() and (candidate / "test").is_dir():
                return candidate
        if here.parent == here:  # reached the filesystem root
            break
        here = here.parent
    return None


def build_manifest(root: Path) -> tuple[pd.DataFrame, list[str]]:
    """Scan every class folder and describe every image file.

    Returns (manifest, issues).

    manifest columns:
        path       -- Path to the file
        split      -- "train" or "test"
        class_name -- the class folder name, e.g. "glioma"
        filename   -- just the file name
        width      -- pixel width
        height     -- pixel height
        mode       -- PIL image mode, e.g. "RGB" or "L"
        is_gray    -- True if the file stores a single colour channel
        bytes      -- file size on disk

    `issues` collects human-readable warnings: files that look like images but
    cannot be opened, and class folders that contain no usable images.

    A corrupt file is EXCLUDED from the manifest rather than stored with null
    dimensions. Reason: every later cell assumes it can open any path in
    `manifest["path"]`, and it reads dimensions -- keeping a broken row would
    crash the plotting cells and quietly skew the counts.
    """
    rows: list[dict] = []
    issues: list[str] = []

    for split in ("train", "test"):
        split_dir = root / split
        if not split_dir.is_dir():
            issues.append(f"Missing split folder: {split_dir}")
            continue

        # Sorted so the manifest order is stable across machines and runs.
        class_dirs = sorted(p for p in split_dir.iterdir() if p.is_dir())

        for class_dir in class_dirs:
            # Counted per class folder, so an *empty* one can be detected below.
            n_usable = 0

            for path in sorted(class_dir.iterdir()):
                if not path.is_file():
                    continue
                if path.suffix.lower() not in IMAGE_EXTENSIONS:
                    continue  # skip .txt, Thumbs.db, etc.

                # PIL opens lazily: it reads the header only, so this stays fast
                # even for thousands of files. A truncated or non-image file
                # raises here, and must not kill the whole notebook.
                try:
                    with Image.open(path) as img:
                        width, height = img.size
                        mode = img.mode
                except Exception as exc:  # noqa: BLE001
                    issues.append(f"Unreadable image skipped: {path} ({exc})")
                    continue

                rows.append(
                    {
                        "path": path,
                        "split": split,
                        "class_name": class_dir.name,
                        "filename": path.name,
                        "width": width,
                        "height": height,
                        "mode": mode,
                        "is_gray": mode in {"L", "1", "I", "F"},
                        "bytes": path.stat().st_size,
                    }
                )
                n_usable += 1

            # An empty class folder is easy to miss and quietly skews every
            # downstream average, so surface it explicitly.
            if n_usable == 0:
                issues.append(
                    f"Class folder has no usable images: {class_dir}  "
                    f"(split={split})"
                )

    return pd.DataFrame(rows), issues


DATA_ROOT = find_data_root(Path.cwd())
ISSUES: list[str] = []

if DATA_ROOT is None:
    print("=" * 70)
    print("NO DATASET FOUND -- every cell below will run but stay empty.")
    print("=" * 70)
    print(
        "Put the data in one of these places (see data/README.md):\n"
        "    data/processed/train/<class>/*.jpg\n"
        "    data/processed/test/<class>/*.jpg\n\n"
        "Quickest way to get a real dataset:\n"
        "    Kaggle -> 'Brain Tumor MRI Dataset' (glioma / meningioma /\n"
        "    pituitary / no_tumor), unzip into data/raw/.\n\n"
        "Then re-run this cell. Nothing else needs editing."
    )
    MANIFEST = pd.DataFrame()
else:
    print(f"Dataset root: {DATA_ROOT}")
    MANIFEST, ISSUES = build_manifest(DATA_ROOT)
    print(f"Indexed {len(MANIFEST)} usable images from {DATA_ROOT}")

    if ISSUES:
        print(f"\n{len(ISSUES)} data issue(s) found -- these are excluded above:")
        for issue in ISSUES:
            print(f"  - {issue}")
    else:
        print("No data issues: every file opened and every class folder is non-empty.")

MANIFEST.head(10)

That `MANIFEST` DataFrame is one row per **usable** image. The columns we care about:

| Column | Meaning |
| --- | --- |
| `split` | `train` or `test` |
| `class_name` | the folder the image came from — this is our **label** |
| `width`, `height` | pixel dimensions, read from the file header |
| `mode` | PIL colour mode; `L` means single-channel grayscale, `RGB` means 3 channels |
| `bytes` | file size on disk |

MRI slices are essentially always grayscale, so if you see `mode == "L"` for most rows
that is expected — not a bug. It is also exactly why `src/data/dataset.py` calls
`img.convert("RGB")`: a 1-channel image has to become 3 channels to feed an
ImageNet-pretrained backbone.

### Data quality is checked up front

The scan cell above reports two kinds of problem, and **excludes** them from the counts
and plots rather than letting them distort the numbers:

- **Unreadable images** — files with an image extension that PIL cannot open
  (truncated downloads, a `.png` that is really text, zero-byte files). Real cause is
  almost always a botched export or an incomplete unzip, so it is worth deleting the
  file and re-downloading rather than working around it.
- **Empty class folders** — a class directory containing no usable images. These are
  genuinely dangerous because they are invisible in a file count: a `no_tumor/` folder
  that is empty makes the model look like it learned a 3-class problem.

Re-read `ISSUES` any time to check the list.

## 2. Look at the images

Numbers only tell you so much. Below we pull a few **random** scans from every class and
draw them side by side.

Things worth actually looking for:

- **Tumours are visible as brighter or darker blobs** in the glioma / meningioma /
  pituitary folders. The `no_tumor` scans should look like plain brain.
- **Is the tumour in the middle?** MRI slices are often cropped tight around the head,
  but badly centred or mostly-black frames are useless training data.
- **Is the whole brain in frame?** Some datasets include partial slices.
- **Do the classes look genuinely different?** If glioma and meningioma scans look
  interchangeable to a human, the model will struggle too — that is a real ceiling on
  accuracy, and no amount of tuning fixes it.

In [ ]:
# ---------------------------------------------------------------------------
# Random sample grid: a few real images per class, drawn as-is (no resizing).
# ---------------------------------------------------------------------------

SAMPLES_PER_CLASS = 8

if MANIFEST.empty:
    print("No data loaded, skipping.")
else:
    classes = sorted(MANIFEST["class_name"].unique())
    n_rows = len(classes)

    fig, axes = plt.subplots(
        n_rows, SAMPLES_PER_CLASS,
        figsize=(SAMPLES_PER_CLASS * 1.9, n_rows * 2.3),
    )

    # Squeeze() keeps axes 2D even when there is only one class or one sample.
    axes = np.atleast_2d(axes)

    for row, class_name in enumerate(classes):
        subset = MANIFEST[MANIFEST["class_name"] == class_name]

        # random_state=SEED so the same pictures come back on every re-run.
        picked = subset.sample(
            n=min(SAMPLES_PER_CLASS, len(subset)),
            random_state=SEED,
        )

        for col, path in enumerate(picked["path"]):
            ax = axes[row, col]
            with Image.open(path) as img:
                ax.imshow(img.convert("L"), cmap="gray")
            ax.set_xticks([])
            ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_edgecolor("0.85")

            # Only label the first column, so captions do not repeat.
            if col == 0:
                ax.set_ylabel(class_name, fontsize=11, fontweight="bold")
                ax.set_title(
                    f"{picked.iloc[col]['width']}x{picked.iloc[col]['height']}",
                    fontsize=8,
                    color="0.4",
                )

    fig.suptitle(
        f"{SAMPLES_PER_CLASS} random scans per class  (seed={SEED})",
        fontsize=13,
        y=1.005,
    )
    fig.tight_layout()
    plt.show()

## 3. How many images per class?

This is the count that decides whether the dataset is usable as-is. We want the totals
*and* the train/test split per class, because a class that is fine overall can still be
badly split (e.g. 400 in train and 3 in test, which makes its test accuracy meaningless).

In [ ]:
# ---------------------------------------------------------------------------
# Counts per class, overall and per split.
# ---------------------------------------------------------------------------

if MANIFEST.empty:
    print("No data loaded, skipping.")
else:
    # One row per class, one column per split, plus a total.
    counts = (
        pd.crosstab(MANIFEST["class_name"], MANIFEST["split"], margins=True, margins_name="TOTAL")
        .sort_values("TOTAL", ascending=False)
    )
    print("Images per class:\n")
    print(counts)

    total = len(MANIFEST)
    print(f"\n{total} images across {MANIFEST['class_name'].nunique()} classes.")

    # Fraction of the dataset each class holds.
    share = (
        MANIFEST["class_name"]
        .value_counts(normalize=True)
        .rename("share")
        .to_frame()
    )
    share["share"] = (share["share"] * 100).round(1)
    share["images"] = MANIFEST["class_name"].value_counts().reindex(share.index)
    print("\nShare of the dataset:\n")
    print(share)

    # Sanity check against the classes this project expects.
    missing = [c for c in EXPECTED_CLASSES if c not in set(MANIFEST["class_name"])]
    unexpected = [
        c for c in sorted(MANIFEST["class_name"].unique()) if c not in EXPECTED_CLASSES
    ]
    if missing:
        print(f"\nMissing expected classes: {missing}")
    if unexpected:
        print(f"\nUnexpected class folders: {unexpected}")

    # Warn about any class folder that is empty or nearly empty.
    tiny = share[share["images"] < 50]
    if not tiny.empty:
        print(f"\nClasses with fewer than 50 images (too few to learn from):\n{tiny}")

## 4. Class distribution

The same counts as a picture, which is far easier to read at a glance:

- **Left** — images per class, split into train vs test. Both bars should be the same
  class, and roughly the same proportion as each other. A class where the test bar is a
  sliver is a warning sign.
- **Right** — each class as a percentage of the whole dataset. This is the view that
  makes imbalance obvious.

In [ ]:
# ---------------------------------------------------------------------------
# Distribution: grouped bar chart + share pie.
# ---------------------------------------------------------------------------

if MANIFEST.empty:
    print("No data loaded, skipping.")
else:
    order = (
        MANIFEST["class_name"].value_counts().index.tolist()
    )  # busiest class first

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.6))

    # --- Left: counts, coloured by split --------------------------------
    sns.countplot(
        data=MANIFEST,
        x="class_name",
        hue="split",
        order=order,
        ax=ax1,
        palette={"train": "#4C72B0", "test": "#DD8452"},
    )
    ax1.set_title("Images per class, by split")
    ax1.set_xlabel("")
    ax1.set_ylabel("Number of images")

    # Put the count on top of each bar so exact numbers are readable.
    for container in ax1.containers:
        ax1.bar_label(container, fontsize=8, padding=2)

    ax1.legend(title="Split", loc="upper right")

    # --- Right: share of the total --------------------------------------
    share = MANIFEST["class_name"].value_counts(normalize=True).reindex(order)
    wedges, _, autotexts = ax2.pie(
        share.values,
        labels=order,
        autopct=lambda p: f"{p:.1f}%",
        startangle=90,
        counterclock=False,
        wedgeprops={"edgecolor": "white", "linewidth": 1.5},
    )
    for autotext in autotexts:
        autotext.set_fontsize(9)
    ax2.set_title("Share of the dataset")

    fig.tight_layout()
    plt.show()

## 5. Image dimensions

Two reasons this matters:

1. **Are the images already uniform?** If everything is already 224x224, the resize step
   in `src/data/dataset.py` is a no-op. If sizes vary wildly, the resize is doing real
   work — and squashing a non-square image distorts the tumour's shape, which can hurt a
   model that is meant to recognise shape.
2. **Do any files not match their header?** A few images lying about their size is a
   sign of a botched export, and those tend to produce confusing training behaviour.

In [ ]:
# ---------------------------------------------------------------------------
# Dimensions: summary, exact-size frequency, and a width-vs-height scatter.
# ---------------------------------------------------------------------------

if MANIFEST.empty:
    print("No data loaded, skipping.")
else:
    sizes = MANIFEST.dropna(subset=["width", "height"]).copy()
    sizes["megapixels"] = (sizes["width"] * sizes["height"]) / 1e6

    print("Resolution summary:\n")
    print(
        sizes[["width", "height", "megapixels"]]
        .describe()
        .loc[["count", "mean", "std", "min", "50%", "max"]]
        .round(2)
        .to_string()
    )

    n_unique = sizes.groupby(["width", "height"]).ngroups
    print(f"\n{n_unique} distinct (width, height) combination(s) in the dataset.")
    print(
        f"Aspect ratio ranges from "
        f"{(sizes['width'] / sizes['height']).min():.2f} to "
        f"{(sizes['width'] / sizes['height']).max():.2f} "
        f"(1.00 = square)."
    )

    already = (sizes["width"] == TARGET_SIZE) & (sizes["height"] == TARGET_SIZE)
    print(
        f"{already.sum()} of {len(sizes)} images ({already.mean():.1%}) are already "
        f"{TARGET_SIZE}x{TARGET_SIZE}; the rest get resized in dataset.py."
    )

    print("\nColour modes:\n")
    print(MANIFEST["mode"].value_counts().to_string())

    # --- Plots ----------------------------------------------------------
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.8))

    # Left: bar chart of every distinct size, capped to the top 15.
    size_counts = (
        sizes.groupby(["width", "height"]).size().sort_values(ascending=False)
    )
    top = size_counts.head(15)
    labels = [f"{w}x{h}" for (w, h) in top.index]
    colours = sns.color_palette("viridis", n_colors=len(top))
    ax1.bar(range(len(top)), top.values, color=colours)
    ax1.set_xticks(range(len(top)))
    ax1.set_xticklabels(labels, rotation=60, ha="right", fontsize=8)
    ax1.set_xlabel("Resolution (width x height)")
    ax1.set_ylabel("Number of images")
    ax1.set_title(
        f"{n_unique} distinct sizes, top {len(top)} shown"
        if n_unique > len(top)
        else "All distinct sizes"
    )

    # Right: scatter, so outlier aspect ratios stand out.
    for class_name in sorted(sizes["class_name"].unique()):
        part = sizes[sizes["class_name"] == class_name]
        ax2.scatter(
            part["width"], part["height"],
            s=18, alpha=0.6, label=class_name,
        )
    ax2.axhline(TARGET_SIZE, ls="--", lw=1, color="0.4")
    ax2.axvline(TARGET_SIZE, ls="--", lw=1, color="0.4")
    ax2.set_xlabel("Width (px)")
    ax2.set_ylabel("Height (px)")
    ax2.set_title(f"Scatter of image sizes (dashed = {TARGET_SIZE}px target)")
    ax2.legend(fontsize=8)

    fig.tight_layout()
    plt.show()

    if n_unique > 1:
        print(
            f"\nNOTE: {n_unique} different resolutions means the resize step will "
            "stretch some images and squeeze others. If most images are not square, "
            "consider a centre-crop or pad-to-square instead of a plain Resize."
        )

## 6. What class imbalance means

**Class imbalance** is when the classes in your dataset do not have the same number of
examples.

A **balanced** dataset looks like this — roughly equal bars:

```
glioma        ############################   700
meningioma    ##########################      690
pituitary     #########################       650
no_tumor      ######################          570
```

An **imbalanced** dataset looks like this:

```
glioma        ########################################   1500
meningioma    ##################                      450
pituitary     ##############                           340
no_tumor      ####                                      100
```

### Why it matters

**Accuracy becomes a lie.** This is the important part. Suppose the dataset is 80%
`glioma`, 10% `meningioma`, 5% `pituitary`, 5% `no_tumor`. A model that ignores the
image entirely and *always* predicts `glioma` scores 80% accuracy — and it has learned
absolutely nothing. When you report "80% accurate", you have learned almost nothing
about the model and a great deal about the dataset.

**It scales with how imbalanced the data is.** At 80/10/5/5 the majority-class baseline
is 80%. Make the split 95/2/2/1 and a useless constant model scores 95%.

**The model is biased toward the common class.** Gradient descent minimises *average*
loss across the batch. If 80% of examples are `glioma`, guessing `glioma` is the
cheapest way to reduce loss, so the model drifts that way. The rare classes get almost
no gradient signal and are predicted poorly.

**On a medical dataset this is the difference between a useful model and a dangerous
one.** If `no_tumor` is the rarest class, that is the class where a false negative is
most costly: telling a patient with no tumour that they have one, or the reverse. The
metric that matters is recall on the rare class, not overall accuracy.

### What to do about it

| Approach | What it does | When to reach for it |
| --- | --- | --- |
| **Stratified split** | Keeps class proportions identical across train/test | Always. Non-negotiable, and nearly free. |
| **Class weights** | Multiplies the loss of rare-class examples | Default fix. Usually recovers most of the gap. |
| **Weighted sampling** (`WeightedRandomSampler`) | Oversamples rare classes instead | When weights alone are not enough. |
| **Augmentation** | Makes more examples of rare classes | Helps, but only if augmentation is realistic for MRI. |
| **Metrics** | Report macro-F1 / per-class recall, not just accuracy | Always report these. Accuracy alone hides the problem. |

The first row is the one to do right now: in `src/data/dataset.py` the class list is
read **once, from `train/`, and reused for `test/`** so the label indices cannot shift
between splits.

In [ ]:
# ---------------------------------------------------------------------------
# Quantify the imbalance on THIS dataset.
# ---------------------------------------------------------------------------

if MANIFEST.empty:
    print("No data loaded, skipping.")
else:
    value_counts = MANIFEST["class_name"].value_counts()
    majority = value_counts.index[0]
    majority_share = value_counts.iloc[0] / len(MANIFEST)
    imbalance_ratio = value_counts.iloc[0] / value_counts.iloc[-1]

    print(f"Most common class : {majority} ({value_counts.iloc[0]} images)")
    print(f"Least common class: {value_counts.index[-1]} ({value_counts.iloc[-1]} images)")
    print(f"Imbalance ratio   : {imbalance_ratio:.1f}x")
    print()
    print(f"Majority-class baseline accuracy: {majority_share:.1%}")
    print(
        "  ^ a model that always predicts "
        f"'{majority}' and never looks at the image scores this much."
    )
    print("    Any model you train must beat that number to be worth anything.")

    # How balanced are the classes, in plain words?
    if imbalance_ratio < 1.5:
        verdict = "BALANCED -- no action needed."
    elif imbalance_ratio < 3:
        verdict = (
            "MILDLY imbalanced -- stratified split + class weights, then move on."
        )
    else:
        verdict = (
            "STRONGLY imbalanced -- report macro-F1 and per-class recall, "
            "use class weights or weighted sampling, and check the rare class "
            "is represented in the test split at all."
        )
    print(f"\nVerdict: {verdict}")

    # A visual sense of how far each class sits from an even split.
    ideal_share = 1 / len(value_counts)
    comparison = pd.DataFrame(
        {
            "images": value_counts,
            "actual_%": (value_counts / len(MANIFEST) * 100).round(1),
            "ideal_%": round(ideal_share * 100, 1),
        }
    )
    comparison["over/under"] = (
        comparison["actual_%"] / comparison["ideal_%"]
    ).round(2)
    print("\nVersus a perfectly even split:\n")
    print(comparison)
    print(
        "\n'over/under' > 1 means the class has more images than an even split "
        "would give it."
    )

## 7. Optional cross-check against `src/`

`src/data/dataset.py` builds the same view of the data at training time. If PyTorch is
installed, this cell confirms the two agree — the notebook convention is to keep
notebooks and `src/` in sync. The `try/except` means the notebook still works on a
machine with no torch.

In [ ]:
# Optional: compare the notebook manifest against the production Dataset.
try:
    from src.data.dataset import BrainTumorDataset, build_transform

    if MANIFEST.empty:
        print("No data loaded, skipping cross-check.")
    else:
        train_ds = BrainTumorDataset(root=DATA_ROOT, split="train")

        print(f"src/data/dataset.py sees {len(train_ds)} train images")
        print(f"this notebook sees          "
              f"{len(MANIFEST[MANIFEST['split'] == 'train'])} train images")
        print(f"classes agree: "
              f"{sorted(train_ds.class_names) == sorted(MANIFEST['class_name'].unique())}")

        # Prove the transform pipeline produces what the model will actually see.
        image, label = train_ds[0]
        print(f"\nAfter build_transform(): shape={tuple(image.shape)}, "
              f"dtype={image.dtype}, min={image.min():.2f}, max={image.max():.2f}")
        print(f"label={label} -> '{train_ds.idx_to_class[label]}'")
        print(
            "Negative minima are expected and correct: Normalize subtracts the "
            "ImageNet mean, so dark pixels land below zero."
        )
except ImportError:
    print("PyTorch is not installed here, so this cross-check is skipped.")
    print("Install it with:  pip install torch torchvision "
          "--index-url https://download.pytorch.org/whl/cpu")

## Summary

| Question | Answer |
| --- | --- |
| What do the scans look like? | Fill in after running the grid above. |
| Counts per class | See the crosstab in section 3. |
| Balanced? | See the verdict in section 6. |
| Uniform image size? | See section 5 — if sizes vary a lot, revisit the resize. |
| What is class imbalance? | A class-count skew that makes plain accuracy misleading. |

### Carry these into the training code

- [ ] Split **stratified by class**, and by **patient** if patient IDs are available
      (`data/README.md` — slicing by slice is the most common way to leak data).
- [ ] Report **macro-F1 and per-class recall** alongside accuracy.
- [ ] Consider **class weights or `WeightedRandomSampler`** if section 6 says imbalanced.
- [ ] Record the seed and these counts so later runs are comparable.

> This project is research/educational. Not a medical device — nothing it outputs may be
> used for diagnosis or treatment decisions.

Before committing:

```powershell
jupyter nbconvert --clear-output --inplace notebooks/01_data_exploration.ipynb
```